In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/cleaned_data.csv')
log_cols = [c for c in df.columns if c.endswith('_log')]

X = df[log_cols]
y = df['behavior_score']

# 80% of data to train the model, 20% held back to honestly test it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

Train size: 120 Test size: 30


In [2]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

baseline = DummyRegressor(strategy='mean').fit(X_train, y_train) # always predicts the average
baseline_pred = baseline.predict(X_test)
print("Baseline MAE:", mean_absolute_error(y_test, baseline_pred))
print("Baseline R2:", r2_score(y_test, baseline_pred)) # will be ~0 or negative — that's expected

Baseline MAE: 6.826666666666666
Baseline R2: -0.024478758580739335


In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rf_pipeline = Pipeline([
('scaler', StandardScaler()),
('rf', RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42))
])
rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)
print("Random Forest MAE:", mean_absolute_error(y_test, rf_pred))
print("Random Forest R2:", r2_score(y_test, rf_pred))

Random Forest MAE: 5.824077846789568
Random Forest R2: 0.17292871593686354


In [4]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
print("XGBoost MAE:", mean_absolute_error(y_test, xgb_pred))
print("XGBoost R2:", r2_score(y_test, xgb_pred))

XGBoost MAE: 5.619667809804281
XGBoost R2: 0.2165245957069737


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/cleaned_data.csv')
log_cols = [c for c in df.columns if c.endswith('_log')]

X = df[log_cols]
y = df['behavior_score']

# 80% of data to train the model, 20% held back to honestly test it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

Train size: 120 Test size: 30


In [2]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

baseline = DummyRegressor(strategy='mean').fit(X_train, y_train) # always predicts the average
baseline_pred = baseline.predict(X_test)
print("Baseline MAE:", mean_absolute_error(y_test, baseline_pred))
print("Baseline R2:", r2_score(y_test, baseline_pred)) # will be ~0 or negative — that's expected

Baseline MAE: 6.826666666666666
Baseline R2: -0.024478758580739335


In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rf_pipeline = Pipeline([
('scaler', StandardScaler()),
('rf', RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42))
])
rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)
print("Random Forest MAE:", mean_absolute_error(y_test, rf_pred))
print("Random Forest R2:", r2_score(y_test, rf_pred))

Random Forest MAE: 5.824077846789568
Random Forest R2: 0.17292871593686354


In [5]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
print("XGBoost MAE:", mean_absolute_error(y_test, xgb_pred))
print("XGBoost R2:", r2_score(y_test, xgb_pred))

XGBoost MAE: 5.619667809804281
XGBoost R2: 0.2165245957069737


In [6]:
%pip install xgboost

  Using cached xgboost-3.3.0-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.3.0-py3-none-win_amd64.whl (69.5 MB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(rf_pipeline, X, y, cv=5, scoring='r2')
print("Cross-validated R2 scores across 5 folds:", cv_scores)
print("Mean R2:", cv_scores.mean(), " Std:", cv_scores.std())

Cross-validated R2 scores across 5 folds: [ 0.21879258  0.15355606 -0.10099874 -0.01957393  0.2024909 ]
Mean R2: 0.09085337300241991  Std: 0.12787840203650555


In [7]:
comparison = pd.DataFrame({
'model': ['Baseline (mean)', 'Random Forest', 'XGBoost'],
'MAE': [mean_absolute_error(y_test, baseline_pred),
mean_absolute_error(y_test, rf_pred),
mean_absolute_error(y_test, xgb_pred)],
'R2': [r2_score(y_test, baseline_pred),
r2_score(y_test, rf_pred),
r2_score(y_test, xgb_pred)]
})
comparison

,model,MAE,R2
0,Baseline (mean),6.826667,-0.024479
1,Random Forest,5.824078,0.172929
2,XGBoost,5.619668,0.216525
